## --Read Bronze Table --

In [0]:
from src.constants import *
from src.common_functions import *
from src.validations import *

import src.common_functions as cf

print(dir(cf))

In [0]:
laborposition_df = spark.table(LABOR_BRONZE_TABLE)

display(laborposition_df)

In [0]:
print(laborposition_df.count())

--Data Validation--

In [0]:
laborposition_df = trim_columns(laborposition_df)

In [0]:
laborposition_df = replace_blank_with_null(laborposition_df)

In [0]:
from pyspark.sql.functions import col

laborposition_df = laborposition_df.withColumn(
    "Labor_Position_Code",
    col("Labor_Position_Code").cast("int")
)

--Remove Duplicates--

In [0]:
laborposition_df = laborposition_df.dropDuplicates(["Labor_Position_Code"])

print("Rows:", laborposition_df.count())

--Null Validation--

In [0]:
from pyspark.sql import functions as F

print("-- Null Validation --")

null_validation_df = laborposition_df.filter(
    F.col("Labor_Position_Code").isNull()
)

print(
    "Rows with NULL Labor_Position_Code:",
    null_validation_df.count()
)

display(null_validation_df)

In [0]:
laborposition_df.printSchema()

In [0]:
display(
    laborposition_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)

In [0]:
print("Silver Labor Position row count:", laborposition_df.count())

--Write to silver --

In [0]:
laborposition_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(LABOR_SILVER_TABLE)

--Validate Silver--

In [0]:
silver_df = spark.table(LABOR_SILVER_TABLE)

print(silver_df.count())

display(silver_df)

In [0]:
silver_check_df = spark.table(
    "databricks_project1.silver.labor_position"
)

print("Silver table count:", silver_check_df.count())

display(
    silver_check_df.filter(
        F.col("Labor_Position_Code") == 9510
    )
)